<a href="https://colab.research.google.com/github/Youseef3550/flyrank-ml-yousef-assighn-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring / Ranking (with a classification model underneath)**

My lane (Refresh / Content Opportunity Scoring) is fundamentally a **ranking**
problem: the output a reviewer needs is an ordered list of pages, not a single
yes/no label. Underneath the ranking, I'll use a **binary classification** model
(page will decline vs. won't) to produce the probability that gets sorted into
that ranking — that's exactly how the starter pipeline's `03_train_model.py`
works (logistic regression / decision tree / random forest, then rank by
predicted probability). So: classification is the mechanism, ranking is the
actual task, because the review capacity (20-50 pages/week) means only the
*order* of the list matters, not whether every single page gets a perfect label.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy target for this stage:** `is_declining_label = trend_direction == "down"`
— the same starter-pipeline proxy documented in the lane guide.

**Where it comes from:** it's a same-window bucket calculated from the last
90 days of impressions/clicks trend, not a verified future outcome. That's why
the lane guide calls it a beginner proxy rather than the ideal capstone target.

**Stronger target I'll move toward by later weeks:** a future-window label —
e.g. features from the prior 90 days predicting decline over the *next* 30 days
— so the model is tested on something that hasn't happened yet, not on a
description of the present.

In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/Youseef3550/flyrank-ml-yousef-assighn-1/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print(f"\nPositive rate (share currently in a 'down' bucket): "
      f"{100*df['is_declining_label'].mean():.1f}%")

df[["content_id", "trend_direction", "is_declining_label", "impressions_90d"]].head(5)

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Positive rate (share currently in a 'down' bucket): 54.2%


,content_id,trend_direction,is_declining_label,impressions_90d
0,content_304f48230142,down,1,3803
1,content_a1fb4e703a9e,down,1,15320
2,content_9aa793d4d895,down,1,12581
3,content_331d6c4de07b,stable,0,11751
4,content_d99b7a2d90ca,down,1,19140


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** (not accuracy).

Why: the reviewer only has capacity for ~50 pages a week, so what matters is
"of the top 50 pages my ranking puts first, how many are truly worth
reviewing?" — not how well the model does across all 30,000 pages. Accuracy
would be misleading here anyway: 54.2% of rows are already labeled "down," so a
model that always predicts "down" would already look decent on accuracy while
being completely useless for ranking.

The starter pipeline documents its own Precision@50 numbers to beat: baseline
rule ≈ 0.24, random forest ≈ 0.74 (`outputs/model_report.md`). That gap — going
from ~12 of the top 50 being right to ~37 of the top 50 being right — is the
number I'm trying to reproduce and then improve on with my own feature/window
choices.

In [10]:
positive_rate = df["is_declining_label"].mean()
print(f"Share of pages currently labeled 'down': {positive_rate:.1%}")
print("A model that always predicts 'down' would already score "
      f"{positive_rate:.1%} accuracy on this label -- useless for a reviewer "
      "with capacity for only ~50 pages/week.\n")

print("Documented starter-pipeline Precision@50 (from outputs/model_report.md):")
print("  baseline hand-rule      : 0.240  (~12 of top 50 correct)")
print("  logistic regression     : 0.400  (~20 of top 50 correct)")
print("  decision tree           : 0.540  (~27 of top 50 correct)")
print("  random forest           : 0.740  (~37 of top 50 correct)")

Share of pages currently labeled 'down': 54.2%
A model that always predicts 'down' would already score 54.2% accuracy on this label -- useless for a reviewer with capacity for only ~50 pages/week.

Documented starter-pipeline Precision@50 (from outputs/model_report.md):
  baseline hand-rule      : 0.240  (~12 of top 50 correct)
  logistic regression     : 0.400  (~20 of top 50 correct)
  decision tree           : 0.540  (~27 of top 50 correct)
  random forest           : 0.740  (~37 of top 50 correct)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [11]:
# One row = one content page (content_id), observed over a 90-day window
cols = ["content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
        "sessions_90d", "avg_position", "ctr", "content_age_days",
        "days_since_last_update", "trend_direction"]

df[cols].head(5)

,content_id,client_id,content_type,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,17,10.6,0.76,187,20,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,9,20.3,0.05,445,25,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,11,36.5,0.09,141,20,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,78,6.2,0.49,463,22,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,145,44.0,0.13,263,14,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement rule (like the starter's `baseline_refresh_score`) can only
combine a handful of signals with hand-picked weights (0.40 visibility + 0.30
freshness + 0.25 position + 0.05 depth gap). It can't learn how those signals
*interact* — e.g. that a stale page only matters if it still has real demand, or
that low CTR only signals a problem at certain position tiers. A quick look at
freshness alone shows why: pages under 90 days since their last update already
split across every trend direction (down, stable, up), so "old = declining"
isn't a clean rule on this data.

The starter pipeline's own numbers make the case directly: the fixed rule gets
Precision@50 ≈ 0.24, while a random forest trained on the same signals reaches
≈ 0.74 — roughly triple the hit rate, using the exact same underlying columns.
That's not because the rule was written badly; it's because the real
relationship between these signals and decline is non-linear and conditional in
a way a few hand-picked weights can't capture, but a model that learns from
thousands of real examples can.

In [12]:
# Quick illustration: freshness alone doesn't cleanly separate declining pages
freshness_bucket = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)

pd.crosstab(freshness_bucket, df["trend_direction"])

trend_direction,down,flat,new,stable,up
days_since_last_update,,,,,
<90d,10576,899,2135,3839,3206
90-180d,5604,237,76,2099,1155
180-365d,79,15,24,24,27
365d+,3,1,1,0,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.